<a href="https://colab.research.google.com/github/GiovanniPioDelvecchio/NLP-Project/blob/GloVePreTrained/HumanValueRecognition_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers
!pip install datasets

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 28.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 KB 4.8 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.8/462.8 KB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 KB 17.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 KB 29.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 KB 21.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3


In [2]:
# Cell for the download of the datasets
!wget https://zenodo.org/record/7550385/files/arguments-training.tsv
!wget https://zenodo.org/record/7550385/files/labels-training.tsv
!wget https://zenodo.org/record/7550385/files/arguments-validation.tsv
!wget https://zenodo.org/record/7550385/files/labels-validation.tsv
!wget https://zenodo.org/record/7550385/files/arguments-test.tsv
!wget https://zenodo.org/record/7550385/files/arguments-validation-zhihu.tsv
!wget https://zenodo.org/record/7550385/files/labels-validation-zhihu.tsv

--2023-01-30 15:11:53--  https://zenodo.org/record/7550385/files/arguments-training.tsv
Resolving zenodo.org (zenodo.org)... 188.185.124.72
Connecting to zenodo.org (zenodo.org)|188.185.124.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1012498 (989K) [application/octet-stream]
Saving to: ‘arguments-training.tsv’

arguments-training. 100%[===================>] 988.77K   512KB/s    in 1.9s    

2023-01-30 15:11:56 (512 KB/s) - ‘arguments-training.tsv’ saved [1012498/1012498]

--2023-01-30 15:11:56--  https://zenodo.org/record/7550385/files/labels-training.tsv
Resolving zenodo.org (zenodo.org)... 188.185.124.72
Connecting to zenodo.org (zenodo.org)|188.185.124.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 253843 (248K) [application/octet-stream]
Saving to: ‘labels-training.tsv’

labels-training.tsv 100%[===================>] 247.89K   513KB/s    in 0.5s    

2023-01-30 15:11:58 (513 KB/s) - ‘labels-training.tsv’ saved [253843/2

In [3]:
# imports for dataset loading
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# torch imports
import torch
import torchtext
from torchtext.data import get_tokenizer
from torchtext.vocab import GloVe
from torch.utils.data import DataLoader
from torchtext.data.functional import to_map_style_dataset
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam

# progress bar
from tqdm import tqdm
# garbage collector
import gc

In [29]:
seed = 10
#torch.use_deterministic_algorithms(False)

In [5]:
def huggingface_from_pandas(pandas_df):
  hf_ds = Dataset.from_pandas(pandas_df, preserve_index=False)
  hf_ds = hf_ds.remove_columns(["Argument ID", "Argument ID2"])
  hf_ds = hf_ds.map(lambda x:{"labels": [int(x[col]) for col in hf_ds.column_names if
                                      col not in ['Conclusion', 'Stance', 'Premise']]})
  label_cols = [col for col in hf_ds.column_names if col not in ['Conclusion', 'Stance', 'Premise', "labels"]]
  hf_ds = hf_ds.remove_columns(label_cols)
  return hf_ds, label_cols

In [6]:
# Dataset loading and splitting
raw_training = pd.read_csv("arguments-training.tsv", encoding='utf-8', sep='\t', header=0)
raw_training_lab = pd.read_csv("labels-training.tsv", encoding='utf-8', sep='\t', header=0)
raw_test = pd.read_csv("arguments-validation.tsv", encoding='utf-8', sep='\t', header=0)
raw_test_lab = pd.read_csv("labels-validation.tsv", encoding='utf-8', sep='\t', header=0)

train = raw_training.join(raw_training_lab,how='inner' ,lsuffix='2') # joining labels
test = raw_test.join(raw_test_lab, how='inner', lsuffix='2') # joining labels
train, val = train_test_split(train ,train_size=.80, random_state=seed) # splitting training

train_ds, label_list = huggingface_from_pandas(train)
val_ds, _ = huggingface_from_pandas(val)
test_ds, _ = huggingface_from_pandas(test)

print(train_ds[0])
print(label_list)

whole_dataset = DatasetDict()
whole_dataset["train"] = train_ds.with_format("torch")
whole_dataset["val"] = val_ds.with_format("torch")
whole_dataset["test"] = test_ds.with_format("torch")

  0%|          | 0/4314 [00:00<?, ?ex/s]

  0%|          | 0/1079 [00:00<?, ?ex/s]

  0%|          | 0/1896 [00:00<?, ?ex/s]

{'Conclusion': 'We should ban the Church of Scientology', 'Stance': 'in favor of', 'Premise': "Scientology is not a true religion it is a sect or a cult which brainwashes it's followers and makes money from them.", 'labels': [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0]}
['Self-direction: thought', 'Self-direction: action', 'Stimulation', 'Hedonism', 'Achievement', 'Power: dominance', 'Power: resources', 'Face', 'Security: personal', 'Security: societal', 'Tradition', 'Conformity: rules', 'Conformity: interpersonal', 'Humility', 'Benevolence: caring', 'Benevolence: dependability', 'Universalism: concern', 'Universalism: nature', 'Universalism: tolerance', 'Universalism: objectivity']


In [7]:
print(whole_dataset.keys())
print(whole_dataset["train"])

dict_keys(['train', 'val', 'test'])
Dataset({
    features: ['Conclusion', 'Stance', 'Premise', 'labels'],
    num_rows: 4314
})


In [8]:
print(whole_dataset["train"]["labels"][0].shape)
print(whole_dataset["train"]["labels"])

torch.Size([20])
tensor([[1, 1, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 1],
        ...,
        [0, 0, 0,  ..., 0, 0, 1],
        [0, 0, 0,  ..., 0, 1, 0],
        [0, 0, 1,  ..., 0, 0, 1]])


In [9]:
# Pretrained GloVe setup

global_vectors = GloVe(name='6B', dim=100)

# the current choice is to give an id to each word
tokenizer = get_tokenizer("basic_english")

embeddings = global_vectors.get_vecs_by_tokens(tokenizer("Hello, How are you?"), lower_case_backup=True)

print(embeddings.shape)

.vector_cache/glove.6B.zip: 862MB [02:40, 5.38MB/s]                           
100%|█████████▉| 399999/400000 [00:14<00:00, 27021.15it/s]


torch.Size([6, 100])


In [227]:
max_words = 25
embed_len = 100
batch_size = 32

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
# collate function where the Premises are tokenized and embedded in batches
def vectorize_batch(batch):
    X = [elem["Premise"] for elem in batch]
    Y = [elem["labels"] for elem in batch]
    X = [tokenizer(x) for x in X]
    X = [tokens+[""] * (max_words-len(tokens))  if len(tokens)<max_words else tokens[:max_words] for tokens in X]
    X_tensor = torch.zeros(len(batch), max_words, embed_len)
    Y_tensor = torch.zeros(len(batch), Y[0].shape[0])
    for i, tokens in enumerate(X):
        X_tensor[i] = global_vectors.get_vecs_by_tokens(tokens)
        Y_tensor[i] = Y[i]
    return X_tensor, Y_tensor

train_dataset = whole_dataset["train"].remove_columns(["Stance", "Conclusion"])
val_dataset = whole_dataset["val"].remove_columns(["Stance", "Conclusion"])
test_dataset = whole_dataset["test"].remove_columns(["Stance", "Conclusion"])
print(val_dataset.shape)

# Construction of the Dataloaders for train and validation
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in vectorize_batch(x)))
val_loader  = DataLoader(val_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in vectorize_batch(x)))
test_loader  = DataLoader(test_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in vectorize_batch(x)))

cuda:0
(1079, 2)


In [316]:
num_classes = 20

# Simple model to perform some tests with pytorch
class EmbeddingClassifier(nn.Module):
    def __init__(self):
        super(EmbeddingClassifier, self).__init__() 
        # Not sure about this
        #self.seq_length = batch_size
        self.input_dim = (batch_size, max_words,embed_len)
        self.num_layers = 1
        self.gru_output_dim = self.input_dim
        self.gru = nn.GRU(input_size = embed_len,
                          hidden_size = embed_len,
                          num_layers = 1,
                          batch_first=True, 
                          bidirectional = True)
        self.flatten = nn.Flatten(start_dim=1)
        self.linear_1 = nn.Linear(max_words*embed_len*2, 512)
        self.relu = nn.ReLU()

        self.linear_2 = nn.Linear(512,256)
        #nn.ReLU(),

        self.linear_3 = nn.Linear(256,128)
        #nn.ReLU(),

        self.linear_4 = nn.Linear(128, 64)
        self.linear_5 = nn.Linear(64, num_classes)
        
                

    def forward(self, X_batch):
        h0 = torch.zeros(2,X_batch.shape[0], embed_len)
        h0 = h0.to(device)
        out, hn = self.gru(X_batch, h0)
        out = self.flatten(out)
        out = self.linear_1(out)
        out = self.relu(out)
        out = self.linear_2(out)
        out = self.relu(out)
        out = self.linear_3(out)
        out = self.relu(out)
        out = self.linear_4(out)
        out = self.relu(out)
        out = self.linear_5(out)
        return out

# Function needed to compute the validation loss and the accuracy
def CalcValLossAndAccuracy(model, loss_fn, val_loader):
    with torch.no_grad():
        Y_shuffled, Y_preds, losses = [],[],[]
        for X, Y in val_loader:
            preds = model(X)
            loss = loss_fn(preds, Y)
            losses.append(loss.item())
            Y_shuffled.append(Y)
            Y_preds.append(preds.argmax(dim=-1))

        Y_shuffled = torch.cat(Y_shuffled)
        Y_preds = torch.cat(Y_preds)

        print("Valid Loss : {:.3f}".format(torch.tensor(losses).mean()))
        #print("Valid Acc  : {:.3f}".format(accuracy_score(Y_shuffled.detach().cpu().numpy(), Y_preds.detach().cpu().numpy())))

# Training function
def TrainModel(model, loss_fn, optimizer, train_loader, val_loader, epochs=10):
    for i in range(1, epochs+1):
        losses = []
        for X, Y in tqdm(train_loader):
            Y_preds = model(X)

            loss = loss_fn(Y_preds, Y)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if i%1==0:
            print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
            CalcValLossAndAccuracy(model, loss_fn, val_loader)

In [317]:
from torchsummary import summary
epochs = 20
learning_rate = 1e-4

loss_fn = nn.BCEWithLogitsLoss()
embed_classifier = EmbeddingClassifier()
optimizer = Adam(embed_classifier.parameters(), lr=learning_rate)

embed_classifier.to(device)
summary(embed_classifier, 
                input_size=(max_words, embed_len), batch_size=batch_size, device="cuda")
#TrainModel(embed_classifier, loss_fn, optimizer, train_loader, val_loader, epochs)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
               GRU-1  [[-1, 25, 200], [-1, 2, 100]]               0
           Flatten-2                 [32, 5000]               0
            Linear-3                  [32, 512]       2,560,512
              ReLU-4                  [32, 512]               0
            Linear-5                  [32, 256]         131,328
              ReLU-6                  [32, 256]               0
            Linear-7                  [32, 128]          32,896
              ReLU-8                  [32, 128]               0
            Linear-9                   [32, 64]           8,256
             ReLU-10                   [32, 64]               0
           Linear-11                   [32, 20]           1,300
Total params: 2,734,292
Trainable params: 2,734,292
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.31


In [ ]:
TrainModel(embed_classifier, loss_fn, optimizer, train_loader, val_loader, epochs)

100%|██████████| 135/135 [00:02<00:00, 65.90it/s]


Train Loss : 0.496
Valid Loss : 0.411


100%|██████████| 135/135 [00:02<00:00, 65.86it/s]


Train Loss : 0.418
Valid Loss : 0.409


100%|██████████| 135/135 [00:02<00:00, 58.21it/s]


...

100%|██████████| 135/135 [00:02<00:00, 66.39it/s]


Train Loss : 0.379
Valid Loss : 0.373


100%|██████████| 135/135 [00:02<00:00, 65.66it/s]


Train Loss : 0.374
Valid Loss : 0.371


 39%|███▊      | 52/135 [00:00<00:01, 67.68it/s]

In [276]:
def MakePredictions(model, loader):
    Y_shuffled, Y_preds = [], []
    for X, Y in loader:
        preds = model(X)
        Y_preds.append(preds)
        #Y_shuffled.append(Y)
    gc.collect()
    Y_preds = torch.cat(Y_preds)
    Y_preds = Y_preds.sigmoid()
    return Y_preds.detach()

Y_preds = MakePredictions(embed_classifier, val_loader)
print(Y_preds)

tensor([[0.0960, 0.0791, 0.1044,  ..., 0.4472, 0.0565, 0.2697],
        [0.0093, 0.1080, 0.0136,  ..., 0.0299, 0.0495, 0.1383],
        [0.0295, 0.1602, 0.0160,  ..., 0.0126, 0.0500, 0.2132],
        ...,
        [0.1161, 0.2986, 0.0096,  ..., 0.0088, 0.0643, 0.1911],
        [0.0370, 0.3167, 0.0798,  ..., 0.1257, 0.0420, 0.0488],
        [0.0990, 0.5122, 0.0550,  ..., 0.0138, 0.2426, 0.1234]],
       device='cuda:0')


In [277]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print(len(Y_preds))
print(len(Y_preds[0]))
print(Y_preds[0])

1079
20
tensor([0.0960, 0.0791, 0.1044, 0.0774, 0.5261, 0.0867, 0.3949, 0.0739, 0.6416,
        0.3765, 0.0192, 0.0946, 0.0438, 0.1006, 0.3402, 0.2097, 0.1101, 0.4472,
        0.0565, 0.2697], device='cuda:0')


In [278]:
def keep_above_thresh(Y_preds, thr):
  Y_preds_thr = np.copy(Y_preds.numpy())
  max_rows = Y_preds_thr.shape[0]
  max_cols = Y_preds_thr.shape[1]
  for i in range(max_rows):
    new_row = np.array([1 if Y_preds_thr[i][j] > thr else 0 for j in range(max_cols)])
    Y_preds_thr[i] = new_row
  return Y_preds_thr

Y_preds_thr = keep_above_thresh(Y_preds.to('cpu'), 0.30)
print(Y_preds[0])
print(Y_preds_thr[0])

tensor([0.0960, 0.0791, 0.1044, 0.0774, 0.5261, 0.0867, 0.3949, 0.0739, 0.6416,
        0.3765, 0.0192, 0.0946, 0.0438, 0.1006, 0.3402, 0.2097, 0.1101, 0.4472,
        0.0565, 0.2697], device='cuda:0')
[0. 0. 0. 0. 1. 0. 1. 0. 1. 1. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0.]


In [279]:
from sklearn.metrics import f1_score
def compute_f1_macro(M_true, M_pred):
    f1_scores = []
    for i in range(M_true.shape[1]):
        true = M_true[:, i]
        pred = M_pred[:, i]
        f1_scores.append(f1_score(true, pred, zero_division=0))
    print(f1_scores)
    return np.mean(f1_scores)

Y_true = val_dataset["labels"]
f1_macro = compute_f1_macro(Y_true, Y_preds_thr)
print(f1_macro)

[0.5244444444444444, 0.48372093023255813, 0.0, 0.0, 0.5802650957290132, 0.2913385826771654, 0.453781512605042, 0.0, 0.6422222222222221, 0.6543665436654367, 0.3626373626373627, 0.3993055555555556, 0.0, 0.0, 0.3699825479930192, 0.10582010582010581, 0.6018755328218244, 0.6206896551724139, 0.23280423280423276, 0.28486646884273]
0.33040603966115634


In [273]:
Y_preds = MakePredictions(embed_classifier, test_loader)
Y_preds_thr = keep_above_thresh(Y_preds.to('cpu'), 0.30)
Y_true = test_dataset["labels"]
f1_macro = compute_f1_macro(Y_true, Y_preds_thr)
print(f1_macro)

[0.358161648177496, 0.446443172526574, 0.0, 0.0, 0.5269943593875906, 0.22591362126245845, 0.3846153846153846, 0.0, 0.6506159014557671, 0.5443234836702954, 0.25098039215686274, 0.45463049579045833, 0.0, 0.0, 0.48769574944071586, 0.04620462046204621, 0.576395242451967, 0.2962962962962963, 0.13240418118466898, 0.21708185053380782]
0.27993781997061956


In [ ]:
entry_to_check = 0
print(np.array(Y_true[entry_to_check].numpy(), dtype="float64"))
print(Y_preds_thr[entry_to_check])

[1. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0.]
[0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]


In [ ]:
from sklearn.metrics import classification_report

Y_true = val_dataset["labels"]
Y_preds_np = Y_preds.numpy()
Y_true_cl = np.argmax(Y_true, axis = 1)
Y_preds_cl = np.argmax(Y_preds_np, axis = 1)


print(classification_report(Y_true_cl, Y_preds_cl, zero_division = 0))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       190
           1       0.00      0.00      0.00       175
           2       0.00      0.00      0.00        16
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00       172
           5       0.00      0.00      0.00        60
           6       0.00      0.00      0.00        34
           7       0.00      0.00      0.00        17
           8       0.14      0.64      0.23       161
           9       0.00      0.00      0.00       120
          10       0.00      0.00      0.00        18
          11       0.00      0.00      0.00        34
          12       0.00      0.00      0.00         5
          13       0.00      0.00      0.00        12
          14       0.00      0.00      0.00        22
          15       0.00      0.00      0.00         7
          16       0.02      0.33      0.03        18
          17       0.00    